In [2]:
import aisuite as ai
import json
import re
import os
import getpass

In [3]:
def load_openai_api_key():
    key = os.environ.get("OPENAI_API_KEY")

    if not key:
        key = getpass.getpass("Inserisci la tua OpenAI API key (input nascosto): ").strip()

    if not key:
        raise RuntimeError("API key mancante. Configura OPENAI_API_KEY e riprova.")

    os.environ["OPENAI_API_KEY"] = key
    return True

load_openai_api_key()
print("API key configurata (il valore non viene mostrato).")



API key configurata (il valore non viene mostrato).


In [5]:
client = ai.Client(
    {
        "openai":{
            "timeout": 100,
            "api_key": os.environ["OPENAI_API_KEY"]
        }   
    }
)

MODEL = "openai:gpt-4o-mini"

In [ ]:
messaggio = (
    "Calcola esattamente 184736.29 * 583.17. "
    "Restituisci il risultato numerico e una breve spiegazione."
)

response_senza_tool = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Sei un assistente utile. Rispondi in italiano."},
        {"role": "user", "content": messaggio},
    ],
    temperature=0.5
)

print(response_senza_tool.choices[0].message.content)

Il risultato della moltiplicazione \( 184736.29 \times 583.17 \) è \( 107,053,303.6793 \).

Per calcolare questo prodotto, si moltiplicano i due numeri seguendo le regole della moltiplicazione. In questo caso, poiché i numeri hanno entrambe le parti decimali, il risultato finale mantiene la parte decimale, risultando in un numero con quattro cifre decimali.


In [5]:
risultato_verificato = 184736.29 * 583.17
print(f"Risultato calcolato da Python: {risultato_verificato:.10f}")

Risultato calcolato da Python: 107732662.2392999977


In [ ]:
import math

def calcolatrice(a: float, b: float, operazione: str) -> dict:
    """Esegue una singola operazione aritmetica tra due numeri.

    Operazioni consentite:
    - somma: a + b
    - sottrazione: a - b
    - moltiplicazione: a * b
    - divisione: a / b

    Args:
        a: Primo numero.
        b: Secondo numero.
        operazione: Una tra 'somma', 'sottrazione',
            'moltiplicazione', 'divisione'.

    Returns:
        Un dizionario con operazione, input e risultato oppure un errore.
    """
    operazioni = {
        "somma": lambda x, y: x + y,
        "sottrazione": lambda x, y: x - y,
        "moltiplicazione": lambda x, y: x * y,
        "divisione": lambda x, y: x / y,
    }

    if operazione not in operazioni:
        return {
            "errore": "Operazione non consentita. Usa somma, sottrazione, moltiplicazione o divisione."
        }

    if operazione == "divisione" and b == 0:
        return {"errore": "Divisione per zero non consentita."}

    try:
        risultato = operazioni[operazione](a, b)
    except (TypeError, ValueError, OverflowError) as exc:
        return {"errore": f"Input non valido: {exc}"}

    if not math.isfinite(risultato):
        return {"errore": "Il risultato non è un numero finito."}

    return {
        "operazione": operazione,
        "a": a,
        "b": b,
        "risultato": risultato
    }

In [7]:
print(calcolatrice(12, 5, "somma"))
print(calcolatrice(12, 5, "sottrazione"))
print(calcolatrice(12, 5, "moltiplicazione"))
print(calcolatrice(12, 5, "divisione"))
print(calcolatrice(12, 0, "divisione"))
print(calcolatrice(12, 5, "potenza"))  # Deve essere rifiutata

{'operazione': 'somma', 'a': 12, 'b': 5, 'risultato': 17}
{'operazione': 'sottrazione', 'a': 12, 'b': 5, 'risultato': 7}
{'operazione': 'moltiplicazione', 'a': 12, 'b': 5, 'risultato': 60}
{'operazione': 'divisione', 'a': 12, 'b': 5, 'risultato': 2.4}
{'errore': 'Divisione per zero non consentita.'}
{'errore': 'Operazione non consentita. Usa somma, sottrazione, moltiplicazione o divisione.'}


Esperimento B — diamo la calcolatrice all'LLM tramite aisuite
La differenza fondamentale è tools=[calcolatrice] insieme a max_turns.
- aisuite ricava lo schema del tool dalla firma e dalla docstring Python.
- Il modello decide se invocare il tool e con quali argomenti.
- aisuite esegue la funzione Python e restituisce il risultato al modello.
- Il modello formula la risposta finale usando il risultato ricevuto.
La funzione Python non viene eseguita dal modello: viene eseguita dall'ambiente Python tramite aisuite.

In [11]:
response_con_tool = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Sei un assistente didattico. "
                "Quando l'utente chiede un calcolo numerico, usa sempre "
                "il tool calcolatrice invece di stimare il risultato a mente. "
                "Puoi usare esclusivamente le quattro operazioni offerte dal tool. "
                "Non inventare risultati: attendi il risultato del tool. "
                "Rispondi soltanto con il risultato, non dare nessuna spiegazione"
            )
        },
        {
            "role": "user",
            "content": (
                "Calcola 184736.29 * 583.17. "
            )
        }
    ],
    tools=[calcolatrice],
    max_turns=3,
    temperature=0
)

print(response_con_tool.choices[0].message.content)

107732662.2393


In [6]:

response_senza_tool = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Sei un assistente utile. Rispondi in italiano."},
        {"role": "user", "content": "Che ore sono?"},
    ],
    temperature=0.5
)

print(response_senza_tool.choices[0].message.content)

Non posso fornire l'ora attuale, ma puoi controllare l'orologio del tuo dispositivo per avere l'informazione precisa. Posso aiutarti con qualcos'altro?


In [8]:
def orologio() -> dict:
    """Restituisce l'ora attuale in formato HH:MM:SS.

    Returns:
        Una stringa in formato HH:MM:SS.
    """
    from datetime import datetime

    ora_attuale = datetime.now()

    return {
        "time": ora_attuale.strftime("%H:%M:%S")
    }

In [11]:
orologio()

{'time': '15:46:47'}

In [ ]:
from datetime import datetime


15:40:59


In [ ]:

print(f"{datetime.now().strftime('%H:%M:%S')}")

In [ ]:
response_con_tool = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Sei un assistente generico. "
                "Se ti chiedono l'ora usa il tool orologio invece di stimare il risultato"
                "Altrimenti rispondi per come puoi"
                #"Quando l'utente chiede un orario numerico, usa sempre il tool orologio"
                #"Non inventare risultati: attendi il risultato del tool. "
                #"Rispondi soltanto con il risultato, non dare nessuna spiegazione"
            )
        },
        {
            "role": "user",
            "content": (
                "Che ore sono?"
            )
        }
    ],
    tools=[orologio],
    max_turns=3,
    temperature=0.99
)

print(response_con_tool.choices[0].message.content)

Napoleone Bonaparte, nato il 15 agosto 1769 in Corsica, fu un importante generale e statista francese, noto per il suo ruolo nel corso della Rivoluzione francese e come Imperatore dei Francesi. Dopo aver raggiunto il potere in Francia, Napoleone instaurò un regime autoritario e implementò numerose riforme, tra cui il Codice civile napoleontico, che ha influenzato i sistemi giuridici di molti paesi.

Nel 1804, Napoleone si proclamò Imperatore dei Francesi e guidò una serie di campagne militari note come le Guerre napoleoniche, espandendo notevolmente il territorio francese in Europa. Tuttavia, il suo impero iniziò a declinare dopo la disastrosa invasione della Russia nel 1812 e fu infine sconfitto nella battaglia di Waterloo nel 1815. Dopo la sconfitta, fu esiliato sull'isola di Sant'Elena, dove morì il 5 maggio 1821.

Napoleone è una figura controversa, visto da alcuni come un grande leader e riformatore, e da altri come un tiranno ambizioso che ha causato enormi sofferenze in Europa.


In [24]:
#domanda = "Che ore sono?"
#domanda = "quanto fa 1456.34 / 34.1 ?"
domanda = "CHe temperatura c'é fuori a Lecce?"

In [25]:
response_con_tool_multipli = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Sei un assistente generico. "
                "Rispondi soltanto con il risultato se usi un tool, altrimenti dai la risposta generica."
            )
        },
        {
            "role": "user",
            "content": domanda
        }
    ],
    tools=[calcolatrice, orologio],
    max_turns=3,
    temperature=0.5
)

print(response_con_tool_multipli.choices[0].message.content)

Non posso fornire informazioni in tempo reale, inclusa la temperatura attuale a Lecce. Ti consiglio di controllare un'app meteo o un sito web affidabile per le informazioni più aggiornate.


In [31]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

def get_weather_data_Lecce():
	"""Fetches weather data from the Open-Meteo API for Lecce.

	Args:
		latitude: Latitude of the location.
		longitude: Longitude of the location

	Returns: 
		A dictionary containing the weather data for Lecce."""

	# Setup the Open-Meteo API client with cache and retry on error
	cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
	retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
	openmeteo = openmeteo_requests.Client(session = retry_session)

	# Make sure all required weather variables are listed here
	# The order of variables in hourly or daily is important to assign them correctly below
	url = "https://api.open-meteo.com/v1/forecast"
	params = {
		"latitude": 52.52,
		"longitude": 13.41,
		"hourly": ["temperature_2m", "rain", "cloud_cover"],
		"current": ["temperature_2m", "rain", "cloud_cover"],
		"forecast_days": 1,
	}
	responses = openmeteo.weather_api(url, params = params)

	# Process first location. Add a for-loop for multiple locations or weather models
	response = responses[0]

	current = response.Current()
	current_temperature_2m = current.Variables(0).Value()
	current_rain = current.Variables(1).Value()
	current_cloud_cover = current.Variables(2).Value()

	return {
		"temperature_2m": current_temperature_2m,
		"rain": current_rain,
		"cloud_cover": current_cloud_cover
	}


In [32]:
domanda = "CHe temperatura c'é fuori a Lecce?"

In [34]:
response_con_tool_multipli = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Sei un assistente generico. "
                "Rispondi soltanto con il risultato se usi un tool, altrimenti dai la risposta generica."
            )
        },
        {
            "role": "user",
            "content": domanda
        }
    ],
    tools=[calcolatrice, orologio, get_weather_data_Lecce],
    max_turns=3,
    temperature=0.5
)

print(response_con_tool_multipli.choices[0].message.content)

La temperatura attuale a Lecce è di circa 5.5°C.
